In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

from pathlib import Path
import json
from functools import partial
import torch
torch.autograd.set_detect_anomaly(True)
from PIL import Image
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from transformers import ProcessorMixin, MllamaProcessor, AutoTokenizer, AutoImageProcessor
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from transformers import TrainingArguments
from transformers import Trainer
from peft import LoraConfig, get_peft_model

from dall_e import map_pixels, unmap_pixels, load_model
from dall_e import Encoder, Decoder

from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from datasets import load_dataset, Dataset

from src.agents.tools.transmission import EncodingTool, DecodingTool
from src.agents.tools.segmentation import SemanticSegmentationTool

from torchmetrics import JaccardIndex

from src.kitti_tracking import KittiDataset
from src.kitti_tracking_hf import KittiHFIterableDataset
from arc_trainer import ArcTrainer
from arc_utils import ARCCollator, ArcProcessor, ExtendedLMHead, ExtendEmbedding

/home/leonard/anaconda3/envs/agentic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/leonard/anaconda3/envs/agentic/lib/python3.10/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [3]:
root_dir = "/mnt/ssd/kitti_tracking"
n_steps, n_pred_steps = 1, 0

dataset_builder = KittiHFIterableDataset(
	root_dir=root_dir,
	split="training",
	n_steps=n_steps,
	n_pred_steps=n_pred_steps,
	transform=T.Compose([
		T.Resize((120, 320)),
	])
)
dataset = dataset_builder.to_hf_dataset()

for sample in dataset: break

In [4]:
model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

vq_enc: Encoder = load_model(f"./checkpoints/encoder.pkl", "cpu")
vq_enc.eval()
for param in vq_enc.parameters(): param.require_grad = False

new_tokens = [
	f"<|vq_{i}|>" for i in range(vq_enc.blocks[-1].conv.w.shape[0])
] + ["<|begin_of_mask|>", "<|end_of_mask|>", "<|image|>"]
    
tokenizer = AutoTokenizer.from_pretrained("./checkpoints", use_fast=True)
tokenizer.bom_token = "<|begin_of_mask|>"
tokenizer.eom_token = "<|end_of_mask|>"
tokenizer.add_tokens(new_tokens)

with open(Path("./checkpoints") / "chat_template.json", "r", encoding="utf-8") as f:
    chat_template = json.load(f)
processor = MllamaProcessor(
    AutoImageProcessor.from_pretrained(model_id),
    tokenizer,
	chat_template=chat_template["chat_template"]
)

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.45it/s]


In [5]:
collator = ARCCollator(
    processor, processor.tokenizer,
    vq_enc
)

In [ ]:

model.config.use_cache = False
# model.resize_token_embeddings(len(tokenizer))
model.lm_head = ExtendedLMHead.from_llama("./checkpoints", model)
model.language_model.embed_tokens = ExtendEmbedding.from_llama("./checkpoints", model)
new_vocab_size = len(processor.tokenizer) - 1
model.config.get_text_config().vocab_size = new_vocab_size
model.vocab_size = new_vocab_size

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'down_proj', 'gate_proj', 'up_proj',
        # "embed_tokens", "lm_head",
    ],
    use_dora=True, # optional DoRA 
    init_lora_weights="gaussian"
)
for p in model.language_model.embed_tokens.base_embedding.parameters(): p.requires_grad = False
for p in model.lm_head.base_head.parameters(): p.requires_grad = False
for p in model.model.vision_model.parameters(): p.requires_grad = False
for p in model.model.multi_modal_projector.parameters(): p.requires_grad = False

for p in model.language_model.embed_tokens.extra_embedding.parameters(): p.requires_grad = True
for p in model.lm_head.base_head.parameters(): p.requires_grad = True

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# llm_processor = AutoProcessor.from_pretrained(model_id)

trainable params: 12,410,880 || all params: 10,749,756,963 || trainable%: 0.1155


In [7]:
training_args = TrainingArguments(
	max_steps=3000,
	output_dir='./results',
	logging_dir='./logs',
	gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  
	per_device_train_batch_size=1,
	per_device_eval_batch_size=1,
	# num_train_epochs=1,
	# logging_steps=10,
	# save_total_limit=2,
	# disable_tqdm=False,       # enable progress bar
    logging_strategy="steps",
    logging_steps=10,          # show every step
    logging_first_step=True,  # show step 0/1
    bf16=True,
    # use_cpu=True,
    remove_unused_columns=False,
	learning_rate=1e-5,
    lr_scheduler_type="cosine",
    max_grad_norm=1,
    gradient_accumulation_steps=8,
)
arc_trainer = Trainer(
	# ckpt_path="",
	model=model,
	args=training_args,
	train_dataset=dataset,
    data_collator=partial(collator, prompts=['people', 'vehicles'], application="autonomous driving"),
)
arc_trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
1,13.641000
10,13.288600
20,12.314600
30,10.947500
40,10.038100
50,8.292400
60,7.951300
70,6.173800
80,3.345000
90,2.857400


TrainOutput(global_step=3000, training_loss=1.6066845531562963, metrics={'train_runtime': 158311.935, 'train_samples_per_second': 0.152, 'train_steps_per_second': 0.019, 'total_flos': 7.694690122628467e+17, 'train_loss': 1.6066845531562963, 'epoch': 2.3333333333333335})

In [8]:
model.save_pretrained("./checkpoints/new")

In [11]:
arc_trainer.save_model("./checkpoints/new")